In [2]:
import os
import pandas as pd
import s3fs
import sklearn

code pour récupérer en distant sur S3

In [3]:
#os.environ['AWS_S3_ENDPOINT']

#S3_ENDPOINT_URL = 'http://' +os.environ['AWS_S3_ENDPOINT']
#S3_ENDPOINT_URL

#fs = s3fs.S3FileSystem(client_kwargs = {'endpoint_url' : S3_ENDPOINT_URL})

#fs.ls('') 

#BUCKET = 'ematzner-ensae'
#FILE_KEY_S3 = '/readmission_avc.parquet'
#FILE_PATH_S3 = BUCKET+ FILE_KEY_S3

#with fs.open(FILE_PATH_S3, mode = 'rb') as file_in : 
#    dataini = pd.read_parquet(file_in)

Code pour lire en local

In [4]:
dataini=pd.read_parquet('data/readmission_avc.parquet')
dataini

,modeEntree,modeSortie,duree,ghm2,dp,sexe,age,nbActe,nbRum,nbda,id,id_D
0,8,9,0,01M37E,I671,2.0,76.0,4,1,NaN,l19,
1,8,8,3,01C061,I652,2.0,77.0,4,1,1.0,s7e,
2,8,7,13,01M303,I634,NaN,NaN,4,1,7.0,23f,
3,8,8,11,01M301,I639,1.0,83.0,4,2,2.0,8oi,None
4,8,6,8,01M303,I635,1.0,71.0,4,1,9.0,otz,ld
...,...,...,...,...,...,...,...,...,...,...,...,...
1695,8,7,1,01M30T,I614,1.0,88.0,4,1,4.0,kjg,
1696,8,6,10,01M303,I635,1.0,81.0,10,3,7.0,gie,my
1697,8,8,8,01M301,I639,1.0,68.0,5,3,6.0,6bl,
1698,8,8,11,01M301,I676,2.0,28.0,16,5,7.0,7m8,


On veut évaluer la proba d'être réadmis (id_D != "")

# 1. Data Preprocessing

In [5]:
dataini.dtypes

modeEntree      int32
modeSortie      int32
duree           int32
ghm2           object
dp             object
sexe          float64
age           float64
nbActe          int32
nbRum           int32
nbda          float64
id             object
id_D           object
dtype: object

In [6]:
dataini.isna().sum()

modeEntree      0
modeSortie      0
duree           0
ghm2            0
dp              0
sexe           20
age            20
nbActe          0
nbRum           0
nbda          134
id              0
id_D          200
dtype: int64

on ne va pas supprimer toutes les lignes avec NA
on supprime les id_D à NA (car on peut rien en faire)
on laisse les lignes à sexe, age et nbda = NA

In [7]:
dataini = dataini.dropna(axis=0, subset=['id_D']).copy()
dataini

,modeEntree,modeSortie,duree,ghm2,dp,sexe,age,nbActe,nbRum,nbda,id,id_D
0,8,9,0,01M37E,I671,2.0,76.0,4,1,NaN,l19,
1,8,8,3,01C061,I652,2.0,77.0,4,1,1.0,s7e,
2,8,7,13,01M303,I634,NaN,NaN,4,1,7.0,23f,
4,8,6,8,01M303,I635,1.0,71.0,4,1,9.0,otz,ld
6,8,7,8,01M302,I639,1.0,92.0,7,2,4.0,np7,
...,...,...,...,...,...,...,...,...,...,...,...,...
1695,8,7,1,01M30T,I614,1.0,88.0,4,1,4.0,kjg,
1696,8,6,10,01M303,I635,1.0,81.0,10,3,7.0,gie,my
1697,8,8,8,01M301,I639,1.0,68.0,5,3,6.0,6bl,
1698,8,8,11,01M301,I676,2.0,28.0,16,5,7.0,7m8,


Colonne réadmission = 1 s'il y a une valeur, 0 sinon

In [8]:
dataini['rea']= (dataini['id_D']!="").astype('int8')
dataini

,modeEntree,modeSortie,duree,ghm2,dp,sexe,age,nbActe,nbRum,nbda,id,id_D,rea
0,8,9,0,01M37E,I671,2.0,76.0,4,1,NaN,l19,,0
1,8,8,3,01C061,I652,2.0,77.0,4,1,1.0,s7e,,0
2,8,7,13,01M303,I634,NaN,NaN,4,1,7.0,23f,,0
4,8,6,8,01M303,I635,1.0,71.0,4,1,9.0,otz,ld,1
6,8,7,8,01M302,I639,1.0,92.0,7,2,4.0,np7,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1695,8,7,1,01M30T,I614,1.0,88.0,4,1,4.0,kjg,,0
1696,8,6,10,01M303,I635,1.0,81.0,10,3,7.0,gie,my,1
1697,8,8,8,01M301,I639,1.0,68.0,5,3,6.0,6bl,,0
1698,8,8,11,01M301,I676,2.0,28.0,16,5,7.0,7m8,,0


In [9]:
dataini.rea.value_counts()

0    1281
1     219
Name: rea, dtype: int64

classe déséquilibrée, mais la rea n'est pas rare non plus. On va sûrement utiliser le metric AUC

Conversion des colonnes en faux numérique (attention, si on convertit en string les NA sont convertis en string et "disparaissent")

In [10]:
str_cols = ['modeEntree', 'modeSortie', 'sexe']
dataini[str_cols] = dataini[str_cols].astype('object')
dataini.isna().sum()

modeEntree      0
modeSortie      0
duree           0
ghm2            0
dp              0
sexe           19
age            19
nbActe          0
nbRum           0
nbda          121
id              0
id_D            0
rea             0
dtype: int64

On regarde si les nbda à NA signifient "manquants" ou "0"

In [11]:
dataini.nbda.value_counts()

3.0     193
4.0     178
2.0     154
1.0     150
5.0     135
6.0     129
7.0     115
8.0      85
9.0      68
10.0     37
11.0     32
13.0     27
12.0     20
14.0     14
15.0     13
16.0      8
17.0      4
18.0      4
26.0      3
19.0      3
23.0      2
27.0      1
24.0      1
21.0      1
20.0      1
22.0      1
Name: nbda, dtype: int64

il n'y a pas de 0, donc les NA signifient sûrement 0

In [12]:
dataini['nbda'] = dataini['nbda'].fillna(0)
dataini

,modeEntree,modeSortie,duree,ghm2,dp,sexe,age,nbActe,nbRum,nbda,id,id_D,rea
0,8,9,0,01M37E,I671,2.0,76.0,4,1,0.0,l19,,0
1,8,8,3,01C061,I652,2.0,77.0,4,1,1.0,s7e,,0
2,8,7,13,01M303,I634,NaN,NaN,4,1,7.0,23f,,0
4,8,6,8,01M303,I635,1.0,71.0,4,1,9.0,otz,ld,1
6,8,7,8,01M302,I639,1.0,92.0,7,2,4.0,np7,,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1695,8,7,1,01M30T,I614,1.0,88.0,4,1,4.0,kjg,,0
1696,8,6,10,01M303,I635,1.0,81.0,10,3,7.0,gie,my,1
1697,8,8,8,01M301,I639,1.0,68.0,5,3,6.0,6bl,,0
1698,8,8,11,01M301,I676,2.0,28.0,16,5,7.0,7m8,,0


il faut supprimer les lignes des patients décédés (modeSortie=9) puisqu'ils ne peuvent pas être réadmis, et dropper 2 colonnes inutiles

In [13]:
dataset = dataini[dataini['modeSortie']!=9].drop(['id', 'id_D'], axis=1)
dataset

,modeEntree,modeSortie,duree,ghm2,dp,sexe,age,nbActe,nbRum,nbda,rea
1,8,8,3,01C061,I652,2.0,77.0,4,1,1.0,0
2,8,7,13,01M303,I634,NaN,NaN,4,1,7.0,0
4,8,6,8,01M303,I635,1.0,71.0,4,1,9.0,1
6,8,7,8,01M302,I639,1.0,92.0,7,2,4.0,0
7,8,8,8,01M301,I638,2.0,88.0,8,2,5.0,0
...,...,...,...,...,...,...,...,...,...,...,...
1695,8,7,1,01M30T,I614,1.0,88.0,4,1,4.0,0
1696,8,6,10,01M303,I635,1.0,81.0,10,3,7.0,1
1697,8,8,8,01M301,I639,1.0,68.0,5,3,6.0,0
1698,8,8,11,01M301,I676,2.0,28.0,16,5,7.0,0


In [14]:
#OUT_PATH_S3 = BUCKET + '/dataset.parquet'

#with fs.open(OUT_PATH_S3, mode='wb') as file_out:
#    dataset.to_parquet(file_out, index=False)

# 2. Feature engineering
## 2.1 Rappel de principes

Il faut standardiser les colonnes numériques, et traiter la colonne sexe

In [15]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, Normalizer

In [16]:
dataset['sexe'].head()

1    2.0
2    NaN
4    1.0
6    1.0
7    2.0
Name: sexe, dtype: object

In [17]:
SimpleImputer(strategy='most_frequent').fit_transform(dataset[['sexe']])

array([[2.0],
       [1.0],
       [1.0],
       ...,
       [1.0],
       [2.0],
       [1.0]], dtype=object)

Transformation arbitraire des NA en 1 (homme, majoritaires). Il faut ensuite encoder en catégoriel.

In [18]:
OneHotEncoder(sparse_output=False, drop="first").fit_transform(SimpleImputer(strategy='most_frequent').fit_transform(dataset[['sexe']]))

array([[1.],
       [0.],
       [0.],
       ...,
       [0.],
       [1.],
       [0.]])

## 2.2 Preprocessing Pipeline

In [19]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, Normalizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [20]:
features = dataset.drop('rea', axis=1)
label = dataset['rea']
features.dtypes

modeEntree     object
modeSortie     object
duree           int32
ghm2           object
dp             object
sexe           object
age           float64
nbActe          int32
nbRum           int32
nbda          float64
dtype: object

In [21]:
num_features = features.select_dtypes([ 'int32', 'float64']).columns
cat_features = features.select_dtypes(['object']).columns

In [22]:
num_features

Index(['duree', 'age', 'nbActe', 'nbRum', 'nbda'], dtype='object')

In [23]:
cat_features

Index(['modeEntree', 'modeSortie', 'ghm2', 'dp', 'sexe'], dtype='object')

In [24]:
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder())
])

In [25]:
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler())
])

In [27]:
preprocessor=ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

In [29]:
preprocessor.fit_transform(features).toarray()

array([[-0.68288188,  0.37261423, -0.29461677, ...,  0.        ,
         0.        ,  1.        ],
       [ 0.35228593,  0.        , -0.29461677, ...,  0.        ,
         1.        ,  0.        ],
       [-0.16529798, -0.02521297, -0.29461677, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-0.16529798, -0.22412657, -0.23119333, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.14525237, -2.87630794,  0.46646456, ...,  0.        ,
         0.        ,  1.        ],
       [-0.78639866,  0.04109156, -0.54831055, ...,  0.        ,
         1.        ,  0.        ]])